In [1]:
# Misc
from warnings import filterwarnings

# Data Wrangling
import re
import numpy as np
import pandas as pd

# Data visualization
from PIL import Image, ImageDraw, ImageFont
import cufflinks as cf
from stylecloud import gen_stylecloud

# Modeling
from keras import metrics
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Embedding
from keras.models import Sequential
from keras.layers import SpatialDropout1D
from keras.callbacks import EarlyStopping, ModelCheckpoint

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

# Preprocessing
import nltk
import unicodedata
from nltk import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Model performance
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score

# Environment setup
cf.go_offline()
nltk.download("all")
filterwarnings("ignore")

# Language detection

from langdetect import detect, DetectorFactory


c:\Users\dcame\AppData\Local\Programs\Python\Python313\Lib\site-packages\stylecloud\stylecloud.py:10: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.

c:\Users\dcame\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning:

In the future `np.object` will be defined as the corresponding NumPy scalar.



[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\dcame\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\dcame\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\dcame\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\dcame\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\dcame\AppData\Roaming\nltk_data...
[

## Funciones

In [2]:
def clean_text(text, pattern="[^a-zA-Z0-9 ]"):
    cleaned_text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore')
    cleaned_text = re.sub(pattern, " ", cleaned_text.decode("utf-8"), flags=re.UNICODE)
    cleaned_text = u' '.join(cleaned_text.lower().split())
    return cleaned_text

In [3]:
def stemming(text, stop_words, stemmer):
    tokens = text.split(" ")
    tokens = [x for x in tokens if x not in stop_words]
    tokens = [stemmer.stem(x) for x in tokens]
    return " ".join(tokens)

In [4]:
def get_wordcloud(text, icon="fa-solid fa-car-side", background_color=None, output_name="./wordcloud.png"):
    # https://fontawesome.com/icons/alicorn?s=solid
    gen_stylecloud(text=text, icon_name=icon, background_color=background_color, output_name=output_name)
    return Image.open(output_name)

In [5]:
DetectorFactory.seed = 0  # para resultados reproducibles

def safe_detect(text):
    text = str(text).strip()
    if not text or len(text.split()) < 3:  # menos de 3 palabras suele fallar
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"


In [6]:
def safe_detect_es_en(text):
    text = str(text).strip()
    if not text or len(text.split()) < 3:
        return "unknown"
    try:
        lang = detect(text)
        if lang in ["es", "en"]:
            return lang
        else:
            return "other"
    except:
        return "unknown"


In [7]:
columns = [
    "CMPLID","ODINO","MFR_NAME","MAKETXT","MODELTXT","YEARTXT","CRASH","FAILDATE","FIRE",
    "INJURED","DEATHS","COMPDESC","CITY","STATE","VIN","DATEA","LDATE","MILES","OCCURENCES",
    "CDESCR","CMPL_TYPE","POLICE_RPT_YN","PURCH_DT","ORIG_OWNER_YN","ANTI_BRAKES_YN",
    "CRUISE_CONT_YN","NUM_CYLS","DRIVE_TRAIN","FUEL_SYS","FUEL_TYPE","TRANS_TYPE","VEH_SPEED",
    "DOT","TIRE_SIZE","LOC_OF_TIRE","TIRE_FAIL_TYPE","ORIG_EQUIP_YN","MANUF_DT","SEAT_TYPE",
    "RESTRAINT_TYPE","DEALER_NAME","DEALER_TEL","DEALER_CITY","DEALER_STATE","DEALER_ZIP",
    "PROD_TYPE","REPAIRED_YN","MEDICAL_ATTN","VEHICLES_TOWED_YN"
]


In [8]:
path = "C:\\Users\\dcame\\Documents\\Diplo CDD\\Modulo 5\\FLAT_CMPL.txt"
df = pd.read_csv(path, sep="\t", encoding="utf-8", on_bad_lines="skip", names=columns)
df.head()

,CMPLID,ODINO,MFR_NAME,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,...,RESTRAINT_TYPE,DEALER_NAME,DEALER_TEL,DEALER_CITY,DEALER_STATE,DEALER_ZIP,PROD_TYPE,REPAIRED_YN,MEDICAL_ATTN,VEHICLES_TOWED_YN
0,1,958241,"Volvo Car USA, LLC",VOLVO,760,1987.0,N,NaN,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
1,2,958130,Ford Motor Company,FORD,THUNDERBIRD,1992.0,N,19941222.0,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
2,3,958132,"Kia America, Inc.",KIA,SEPHIA,1994.0,Y,19941230.0,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
3,4,958133,"Chrysler (FCA US, LLC)",DODGE,600,1987.0,N,19941231.0,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N
4,5,958137,"Chrysler (FCA US, LLC)",DODGE,CARAVAN,1991.0,N,19941218.0,N,0,...,NaN,NaN,NaN,NaN,NaN,NaN,V,NaN,N,N


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2192208 entries, 0 to 2192207
Data columns (total 49 columns):
 #   Column             Dtype  
---  ------             -----  
 0   CMPLID             int64  
 1   ODINO              int64  
 2   MFR_NAME           object 
 3   MAKETXT            object 
 4   MODELTXT           object 
 5   YEARTXT            float64
 6   CRASH              object 
 7   FAILDATE           float64
 8   FIRE               object 
 9   INJURED            int64  
 10  DEATHS             int64  
 11  COMPDESC           object 
 12  CITY               object 
 13  STATE              object 
 14  VIN                object 
 15  DATEA              object 
 16  LDATE              object 
 17  MILES              float64
 18  OCCURENCES         object 
 19  CDESCR             object 
 20  CMPL_TYPE          object 
 21  POLICE_RPT_YN      object 
 22  PURCH_DT           float64
 23  ORIG_OWNER_YN      object 
 24  ANTI_BRAKES_YN     object 
 25  CRUISE_CONT_YN    

"CMPLID", "ODINO", "MFR_NAME", "CITY", "STATE", "VIN", "DATEA", "LDATE", "POLICE_RPT_YN", "ORIG_OWNER_YN", "ANTI_BRAKES_YN", "CRUISE_CONT_YN", "LOC_OF_TIRE", "DOT","TIRE_SIZE","LOC_OF_TIRE","TIRE_FAIL_TYPE","ORIG_EQUIP_YN","MANUF_DT","SEAT_TYPE",
    "RESTRAINT_TYPE","DEALER_NAME","DEALER_TEL","DEALER_CITY","DEALER_STATE","DEALER_ZIP"

In [10]:
cols_to_drop = ["CMPLID", "ODINO", "MFR_NAME", "CITY", "STATE", "VIN", "DATEA", "LDATE", "POLICE_RPT_YN", "ORIG_OWNER_YN", "ANTI_BRAKES_YN", "CRUISE_CONT_YN", "LOC_OF_TIRE", "DOT","TIRE_SIZE","LOC_OF_TIRE","TIRE_FAIL_TYPE","ORIG_EQUIP_YN","MANUF_DT","SEAT_TYPE",
    "RESTRAINT_TYPE","DEALER_NAME","DEALER_TEL","DEALER_CITY","DEALER_STATE","DEALER_ZIP", "REPAIRED_YN", "PURCH_DT","ORIG_OWNER_YN","ANTI_BRAKES_YN",
    "CRUISE_CONT_YN","NUM_CYLS","DRIVE_TRAIN","FUEL_SYS","FUEL_TYPE","TRANS_TYPE", "CMPL_TYPE", "VEH_SPEED"]

In [11]:
df.drop(cols_to_drop, axis=1, inplace=True)
df.head()

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN
0,VOLVO,760,1987.0,N,NaN,N,0,0,ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIA...,NaN,NaN,RADIATOR FAILED @ HIGHWAY SPEED OBSTRUCTING DR...,V,N,N
1,FORD,THUNDERBIRD,1992.0,N,19941222.0,N,0,0,"FUEL SYSTEM, GASOLINE:DELIVERY",NaN,1.0,"FUEL LEAKED FROM FUEL TANK AREA, EMITTING STRO...",V,N,N
2,KIA,SEPHIA,1994.0,Y,19941230.0,N,0,0,POWER TRAIN:AUTOMATIC TRANSMISSION,NaN,NaN,SHIFTED INTO REVERSE VEHICLE JERKED VIOLENTLY....,V,N,N
3,DODGE,600,1987.0,N,19941231.0,N,0,0,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY",NaN,NaN,FUEL TANK ; LEAKS BECAUSE OF RUST GAS LEAK BY ...,V,N,N
4,DODGE,CARAVAN,1991.0,N,19941218.0,N,0,0,SEATS,NaN,1.0,"DRIVER SIDE SEAT FRAME BROKE IN TWO, CAUSING S...",V,N,N


In [12]:
df.isnull().mean() * 100

MAKETXT               0.001095
MODELTXT              0.001186
YEARTXT               0.001095
CRASH                 0.000000
FAILDATE              5.023246
FIRE                  0.000000
INJURED               0.000000
DEATHS                0.000000
COMPDESC              0.006797
MILES                48.859962
OCCURENCES           55.655941
CDESCR                0.004516
PROD_TYPE             0.000912
MEDICAL_ATTN          0.000046
VEHICLES_TOWED_YN     3.253797
dtype: float64

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2192208 entries, 0 to 2192207
Data columns (total 15 columns):
 #   Column             Dtype  
---  ------             -----  
 0   MAKETXT            object 
 1   MODELTXT           object 
 2   YEARTXT            float64
 3   CRASH              object 
 4   FAILDATE           float64
 5   FIRE               object 
 6   INJURED            int64  
 7   DEATHS             int64  
 8   COMPDESC           object 
 9   MILES              float64
 10  OCCURENCES         object 
 11  CDESCR             object 
 12  PROD_TYPE          object 
 13  MEDICAL_ATTN       object 
 14  VEHICLES_TOWED_YN  object 
dtypes: float64(3), int64(2), object(10)
memory usage: 250.9+ MB


In [14]:
#Only first 4 values from FAILDATE column
df["FAILDATE"] = df["FAILDATE"].fillna(0)
df["FAILDATE"] = df["FAILDATE"].apply(lambda x: int(str(int(x))[:4]))


In [15]:
df

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN
0,VOLVO,760,1987.0,N,0,N,0,0,ENGINE AND ENGINE COOLING:COOLING SYSTEM:RADIA...,NaN,NaN,RADIATOR FAILED @ HIGHWAY SPEED OBSTRUCTING DR...,V,N,N
1,FORD,THUNDERBIRD,1992.0,N,1994,N,0,0,"FUEL SYSTEM, GASOLINE:DELIVERY",NaN,1.0,"FUEL LEAKED FROM FUEL TANK AREA, EMITTING STRO...",V,N,N
2,KIA,SEPHIA,1994.0,Y,1994,N,0,0,POWER TRAIN:AUTOMATIC TRANSMISSION,NaN,NaN,SHIFTED INTO REVERSE VEHICLE JERKED VIOLENTLY....,V,N,N
3,DODGE,600,1987.0,N,1994,N,0,0,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY",NaN,NaN,FUEL TANK ; LEAKS BECAUSE OF RUST GAS LEAK BY ...,V,N,N
4,DODGE,CARAVAN,1991.0,N,1994,N,0,0,SEATS,NaN,1.0,"DRIVER SIDE SEAT FRAME BROKE IN TWO, CAUSING S...",V,N,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2192203,HYUNDAI,KONA,2021.0,N,2026,N,0,0,ENGINE,NaN,NaN,VIN: [XXX] 2021 Hyundai Kona experienced sudd...,V,N,N
2192204,HONDA,PILOT,2018.0,N,2025,N,0,0,POWER TRAIN,NaN,NaN,Transfer case exploded internally and cracked ...,V,N,N
2192205,ACURA,TLX,2021.0,N,2025,N,0,0,POWER TRAIN,NaN,NaN,The All Wheel Drive (AWD) system of the vehicl...,V,N,N
2192206,TOYOTA,COROLLA,2023.0,N,2025,N,0,0,ELECTRICAL SYSTEM,NaN,NaN,I am experiencing persistent and unresolved is...,V,N,N


In [16]:
# Valores únicos
unique_values = df["MAKETXT"].unique().tolist()
# Mostrar todos
print(unique_values)



['VOLVO', 'FORD', 'KIA', 'DODGE', 'HYUNDAI', 'PLYMOUTH', 'GERRY', 'NISSAN', 'CHEVROLET', 'MERCURY', 'MAZDA', 'GEO', 'EAGLE', 'JEEP', 'HONDA', 'LEXUS', 'GMC', 'LINCOLN', 'PONTIAC', 'TOYOTA', 'EVENFLO', 'RENAULT', 'SUBARU', 'OLDSMOBILE', 'CADILLAC', 'BUICK', 'CHRYSLER', 'MERCEDES BENZ', 'RENOLUX', 'BMW', 'SATURN', 'YAMAHA', 'COSCO', 'MITSUBISHI', 'UNIROYAL', 'ALFA ROMEO', 'HARLEY-DAVIDSON', 'VOLKSWAGEN', 'SUZUKI', 'INFINITI', 'ISUZU', 'KOLCRAFT', 'MICHELIN', 'UNKNOWN MANUFACTURER', 'MONACO COACH', 'FLEETWOOD', 'AUDI', 'MERKUR', 'PEUGEOT', 'DELCO', 'COACHMEN', 'ACURA', 'GOODYEAR', 'ROCKWOOD', 'CENTURY', 'DUCATI', 'UD', 'FISHER PRICE', 'ITASCA', 'DUNLOP', 'HONORBUILT', 'JOHN DEERE', 'GENERAL', 'GEORGIE BOY', 'SAAB', 'KAWASAKI', 'WINNEBAGO', 'BELL', 'BRIDGESTONE', 'GULF STREAM', 'SKYLINE', 'EMERGENCY ONE', 'WHITE TRUCK', 'LAND ROVER', 'TIFFIN', 'KELLY SPRINGFIELD', 'B.F.GOODRICH', 'AIRSTREAM', 'FREIGHTLINER', 'FIRESTONE', 'NU-WAY', 'KING OF THE ROAD', 'UNKNOWN', 'DOLPHIN', 'DAIHATSU', 'JAGU

In [17]:
top_brands = [
    "NISSAN","CHEVROLET","VOLKSWAGEN","TOYOTA","KIA",
    "HYUNDAI","MAZDA","HONDA","SUZUKI","FORD"
]

df = df[df["MAKETXT"].str.upper().isin(top_brands)].reset_index(drop=True)


In [18]:
df["MODELTXT"].unique()

array(['THUNDERBIRD', 'SEPHIA', 'AEROSTAR', ..., 'ENGINE BLOCK HEATER',
       'MX-30', 'XR150L'], shape=(1232,), dtype=object)

In [19]:
models = [
    # TOYOTA
    "4RUNNER", "TACOMA", "TUNDRA", "CAMRY", "CAMRY HYBRID",
    "C-HR", "COROLLA", "COROLLA CROSS", "COROLLA CROSS HYBRID",
    "COROLLA HATCHBACK", "COROLLA HYBRID", "COROLLA MATRIX",
    "FJ CRUISER", "GR86", "GR COROLLA", "HIGHLANDER", "HIGHLANDER HYBRID",
    "LAND CRUISER", "LAND CRUISER HYBRID", "MATRIX",
    "PRIUS", "PRIUS C", "RAV4", "RAV4 EV", "RAV4 HYBRID", "SEQUOIA",
    "SEQUOIA HYBRID", "SIENNA", "SIENNA HYBRID", "SUPRA", "YARIS",

    # HONDA
    "ACCORD", "ACCORD HYBRID", "CIVIC", "CIVIC HYBRID", "CIVIC TYPE R", "CR-V", "CR-V HYBRID", "CR-Z",
    "CROSSTOUR", "ELEMENT", "FIT", "HR-V", "INSIGHT", "ODYSSEY",
    "PASSPORT", "PILOT", "PRELUDE", "PROLOGUE", "RIDGELINE",

    # NISSAN
    "350Z", "370Z", "ALTIMA", "GT-R", "JUKE", "KICKS", "LEAF", "MAXIMA",
    "MURANO", "PATHFINDER","QUEST", "ROGUE", "ROGUE SPORT", "SENTRA", "VERSA",
    "VERSA NOTE", "XTERRA", "Z", "200SX",

    # VOLKSWAGEN
    "ARTEON", "ATLAS", "ATLAS CROSS SPORT", "BEETLE","GOLF", "GTI","JETTA", "JETTA GLI",
    "NEW BEETLE", "PASSAT", "POLO 1.6 COMFORTLINE", "R32", "ROUTAN", "TAIGUN",
    "TAOS", "TIGUAN", "TOUAREG",

    # MAZDA
    "CX-3", "CX-30", "CX-5", "CX-7", "CX-9",
    "CX9", "CX-50", "CX-70", "CX-90", "MAZDA2", "MAZDA3", "MAZDA5",
    "MAZDA6", "MAZDASPEED3", "MAZDASPEED6", "MX-5",

    # HYUNDAI
    "ACCENT", "ELANTRA", "IONIQ", "PALISADE", "SANTA CRUZ", "SANTA FE",
    "SONATA", "TUCSON",

    # KIA
    "FORTE", "NIRO", "OPTIMA", "RIO", "SEDONA", "SELTOS", "SORENTO",
    "SOUL", "SPORTAGE", "STINGER", "TELLURIDE",

    # CHEVROLET
    "ASTRO", "AVEO", "BLAZER", "CAMARO", "CAPRICE", "CAPTIVA", "CAVALIER",
    "CORVETTE", "CORVETTE Z06", "CORVETTE ZR1",
    "CRUZE", "EQUINOX", "IMPALA", "MALIBU",
    "MONTE CARLO", "ONIX", "OPTRA", "SONIC", "SPARK",
    "SPARK EV", "SUBURBAN", "TAHOE", "TRACKER", "TRAILBLAZER", "TRAVERSE", "VENTURE", "YUKON",
    "SILVERADO 1500", "SILVERADO", "COLORADO", "AVALANCHE",
    "SIERRA",

    # FORD
    "AEROSTAR", "BRONCO", "BRONCO SPORT", "ECO SPORT", "ECOSPORT", "EDGE",
    "ESCAPE", "EXPEDITION", "EXPLORER", "EXCURSION", "FIESTA",
    "FLEX", "FOCUS",    "FREESTAR", "FUSION", "FORD GT", "GT500", "MUSTANG", "SHADOW", "TAURUS",
    "THUNDERBIRD", "WINDSTAR", "ESCORT", "F-150",
    "RANGER", "MAVERICK",

    # SUZUKI
    "GRAND VITARA", "KIZASHI", "SWIFT", "VITARA",
]

In [20]:
df = df[df["MODELTXT"].str.upper().isin(models)].reset_index(drop=True)


In [21]:
df = df[(df["YEARTXT"] >= 2000) & (df["YEARTXT"] <= 2027)].reset_index(drop=True)
df = df[(df["FAILDATE"] >= 2000) & (df["FAILDATE"] <= 2027)].reset_index(drop=True)
df["AGE"] = df["FAILDATE"] - df["YEARTXT"]
df = df[df["AGE"] >= -1].reset_index(drop=True)
df["AGE"] = abs(df["AGE"])
df


,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN,AGE
0,FORD,EXPEDITION,2008.0,Y,2009,N,0,0,VISIBILITY:POWER WINDOW DEVICES AND CONTROLS,NaN,1.0,EXPERIENCED PROBLEM WITH POWER WINDOWS FAILING...,V,N,N,1.0
1,FORD,EXPEDITION,2008.0,Y,2009,N,0,0,VEHICLE SPEED CONTROL,NaN,1.0,EXPERIENCED PROBLEM WITH POWER WINDOWS FAILING...,V,N,N,1.0
2,FORD,RANGER,2000.0,N,2000,N,0,0,POWER TRAIN,NaN,12.0,"I BOUGHT MY TRUCK ON DEC.10,1999.ON DEC.21,WHI...",V,N,N,0.0
3,CHEVROLET,MONTE CARLO,2000.0,N,2000,N,0,0,TIRES,NaN,0.0,WHILE DRIVING THE VEHICLE A TIRE BLEW OUT INTO...,V,N,N,0.0
4,HONDA,ODYSSEY,2000.0,N,2000,N,0,0,STRUCTURE:BODY:DOOR,NaN,999.0,ELECTRIC SLIDING POWER DOORS DO NOT REBOUND WH...,V,N,N,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819939,HONDA,ACCORD,2025.0,N,2026,N,0,0,LANE DEPARTURE: WARNING,NaN,NaN,"Around 20,000 miles my car began having faint ...",V,N,N,1.0
819940,TOYOTA,HIGHLANDER,2017.0,N,2026,N,0,0,POWER TRAIN,NaN,NaN,Power training went out I-70 in warren county ...,V,N,N,9.0
819941,HONDA,PILOT,2018.0,N,2025,N,0,0,POWER TRAIN,NaN,NaN,Transfer case exploded internally and cracked ...,V,N,N,7.0
819942,TOYOTA,COROLLA,2023.0,N,2025,N,0,0,ELECTRICAL SYSTEM,NaN,NaN,I am experiencing persistent and unresolved is...,V,N,N,2.0


In [22]:
df.sort_values("AGE", ascending=False).head()

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN,AGE
816405,CHEVROLET,SILVERADO 1500,2000.0,N,2026,N,0,0,ENGINE,NaN,NaN,Airbag recall Recall Engine knock sensors bad,V,N,N,26.0
816404,CHEVROLET,SILVERADO 1500,2000.0,N,2026,N,0,0,AIR BAGS,NaN,NaN,Airbag recall Recall Engine knock sensors bad,V,N,N,26.0
815191,TOYOTA,TACOMA,2000.0,N,2026,N,0,0,UNKNOWN OR OTHER,NaN,NaN,Broken frame due to rust,V,N,N,26.0
817917,TOYOTA,TUNDRA,2000.0,N,2026,N,0,0,SUSPENSION,51000.0,NaN,The contact owns a 2000 Toyota Tundra. The con...,V,N,N,26.0
817230,TOYOTA,CAMRY,2000.0,N,2026,N,0,0,SEAT BELTS:FRONT:RETRACTOR,177600.0,NaN,The contact owns a 1997 N/A Toyota Camry. The ...,V,N,N,26.0


In [23]:
df["AGE"].describe()

count    819944.000000
mean          5.212708
std           3.873578
min           0.000000
25%           2.000000
50%           5.000000
75%           8.000000
max          26.000000
Name: AGE, dtype: float64

In [24]:
df

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN,AGE
0,FORD,EXPEDITION,2008.0,Y,2009,N,0,0,VISIBILITY:POWER WINDOW DEVICES AND CONTROLS,NaN,1.0,EXPERIENCED PROBLEM WITH POWER WINDOWS FAILING...,V,N,N,1.0
1,FORD,EXPEDITION,2008.0,Y,2009,N,0,0,VEHICLE SPEED CONTROL,NaN,1.0,EXPERIENCED PROBLEM WITH POWER WINDOWS FAILING...,V,N,N,1.0
2,FORD,RANGER,2000.0,N,2000,N,0,0,POWER TRAIN,NaN,12.0,"I BOUGHT MY TRUCK ON DEC.10,1999.ON DEC.21,WHI...",V,N,N,0.0
3,CHEVROLET,MONTE CARLO,2000.0,N,2000,N,0,0,TIRES,NaN,0.0,WHILE DRIVING THE VEHICLE A TIRE BLEW OUT INTO...,V,N,N,0.0
4,HONDA,ODYSSEY,2000.0,N,2000,N,0,0,STRUCTURE:BODY:DOOR,NaN,999.0,ELECTRIC SLIDING POWER DOORS DO NOT REBOUND WH...,V,N,N,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819939,HONDA,ACCORD,2025.0,N,2026,N,0,0,LANE DEPARTURE: WARNING,NaN,NaN,"Around 20,000 miles my car began having faint ...",V,N,N,1.0
819940,TOYOTA,HIGHLANDER,2017.0,N,2026,N,0,0,POWER TRAIN,NaN,NaN,Power training went out I-70 in warren county ...,V,N,N,9.0
819941,HONDA,PILOT,2018.0,N,2025,N,0,0,POWER TRAIN,NaN,NaN,Transfer case exploded internally and cracked ...,V,N,N,7.0
819942,TOYOTA,COROLLA,2023.0,N,2025,N,0,0,ELECTRICAL SYSTEM,NaN,NaN,I am experiencing persistent and unresolved is...,V,N,N,2.0


In [25]:
df.isnull().mean() * 100

MAKETXT               0.000000
MODELTXT              0.000000
YEARTXT               0.000000
CRASH                 0.000000
FAILDATE              0.000000
FIRE                  0.000000
INJURED               0.000000
DEATHS                0.000000
COMPDESC              0.003415
MILES                36.787390
OCCURENCES           64.680881
CDESCR                0.004878
PROD_TYPE             0.000000
MEDICAL_ATTN          0.000000
VEHICLES_TOWED_YN     0.003049
AGE                   0.000000
dtype: float64

In [26]:
imputer = SimpleImputer(strategy="mean")
df[["MILES"]] = imputer.fit_transform(df[["MILES"]])

In [27]:
# Add a helper column with the length of COMPSEC
df["COMPDESC_len"] = df["COMPDESC"].astype(str).str.len()

# For each CDESCR, keep the row with the maximum COMPSEC length
df_no_dupes = df.loc[df.groupby("CDESCR")["COMPDESC_len"].idxmax()]



In [28]:
df_no_dupes.reset_index(drop=True, inplace=True)
df_no_dupes

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN,AGE,COMPDESC_len
0,FORD,EXPEDITION,2002.0,N,2013,N,0,0,SUSPENSION,111086.000000,1.0,...,V,N,N,11.0,10
1,TOYOTA,CAMRY,2002.0,N,2005,N,0,0,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY:FI...",38500.000000,1.0,...,V,N,N,3.0,63
2,KIA,SORENTO,2019.0,N,2023,N,0,0,FUEL/PROPULSION SYSTEM,71775.184051,NaN,...,V,N,N,4.0,22
3,KIA,SORENTO,2004.0,N,2008,N,0,0,ENGINE AND ENGINE COOLING:ENGINE,61500.000000,1.0,...,V,N,N,4.0,32
4,FORD,F-150,2005.0,Y,2005,N,0,0,"SERVICE BRAKES, HYDRAULIC",2000.000000,1.0,...,V,N,N,0.0,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608047,VOLKSWAGEN,ATLAS,2019.0,N,2025,N,0,0,EXTERIOR LIGHTING,71775.184051,NaN,●Vehicle tries to pull me into a completely di...,V,N,N,6.0,17
608048,NISSAN,ROGUE,2023.0,N,2023,N,0,0,ELECTRICAL SYSTEM,71775.184051,NaN,⚠️ Warning Alert continues to display on dashb...,V,N,N,0.0,17
608049,TOYOTA,CAMRY,2012.0,N,2022,N,0,0,AIR BAGS,71775.184051,NaN,我,V,N,N,10.0,8
608050,HONDA,ACCORD HYBRID,2023.0,N,2023,N,0,0,LANE DEPARTURE: WARNING,71775.184051,NaN,车辆系统行驶中死机故障。黑屏。重启消除，又发生。经销商无法解决。,V,N,N,0.0,23


In [29]:
df_no_dupes['CDESCR'].duplicated().sum()


np.int64(0)

In [30]:
df.isnull().mean() * 100

MAKETXT               0.000000
MODELTXT              0.000000
YEARTXT               0.000000
CRASH                 0.000000
FAILDATE              0.000000
FIRE                  0.000000
INJURED               0.000000
DEATHS                0.000000
COMPDESC              0.003415
MILES                 0.000000
OCCURENCES           64.680881
CDESCR                0.004878
PROD_TYPE             0.000000
MEDICAL_ATTN          0.000000
VEHICLES_TOWED_YN     0.003049
AGE                   0.000000
COMPDESC_len          0.000000
dtype: float64

### Limpieza de comentarios

In [31]:
df_no_dupes["CDESCR"] = df_no_dupes["CDESCR"].astype(str).apply(lambda x: clean_text(x))

In [32]:
index = 608048
df_no_dupes.iloc[index, df_no_dupes.columns.get_loc('CDESCR')]



'warning alert continues to display on dashboard after several months of recently purchasing my vehicle for check position of shift lever the dealership had the vehicle for an entire week but failed to fix the issue now continues to inform me that they are waiting on a part to come in but that i can continue to drive my car without it being repaired'

In [33]:
stop_words = [clean_text(x) for x in stopwords.words("english", "spanish")]

In [34]:
df_no_dupes["CDESCR"] = df_no_dupes["CDESCR"].astype(str).map(lambda sentence: " ".join([word for word in sentence.split() if word not in stop_words]))

In [35]:
df_no_dupes.iloc[index, df_no_dupes.columns.get_loc('CDESCR')]

'warning alert continues display dashboard several months recently purchasing vehicle check position shift lever dealership vehicle entire week failed fix issue continues inform waiting part come continue drive car without repaired'

In [36]:
#1
df_no_dupes_100 = df_no_dupes.head(100000).copy()
df_no_dupes_100["LANG"] = df_no_dupes_100["CDESCR"].astype(str).apply(lambda x: safe_detect_es_en(x))

In [37]:
#Range index 100001:200000
df_no_dupes_200 = df_no_dupes.iloc[100000:150000].copy()
df_no_dupes_200["LANG"] = df_no_dupes_200["CDESCR"].astype(str).apply(lambda x: safe_detect_es_en(x))


In [38]:
#Range index 100001:200000
df_no_dupes_201 = df_no_dupes.iloc[150000:200000].copy()
df_no_dupes_201["LANG"] = df_no_dupes_201["CDESCR"].astype(str).apply(lambda x: safe_detect_es_en(x))


In [39]:
#3
df_no_dupes_300 = df_no_dupes.iloc[200001:300000].copy()
df_no_dupes_300["LANG"] = df_no_dupes_300["CDESCR"].astype(str).apply(lambda x: safe_detect_es_en(x))


In [40]:
#4
df_no_dupes_400 = df_no_dupes.iloc[300001:400000].copy()
df_no_dupes_400["LANG"] = df_no_dupes_400["CDESCR"].astype(str).apply(lambda x: safe_detect_es_en(x))


In [43]:
#Filter only Spanish and English
df_no_dupes_100 = df_no_dupes_100[df_no_dupes_100["LANG"].isin(["es", "en"])].reset_index(drop=True)
df_no_dupes_200 = df_no_dupes_200[df_no_dupes_200["LANG"].isin(["es", "en"])].reset_index(drop=True)
df_no_dupes_201 = df_no_dupes_201[df_no_dupes_201["LANG"].isin(["es", "en"])].reset_index(drop=True)
df_no_dupes_300 = df_no_dupes_300[df_no_dupes_300["LANG"].isin(["es", "en"])].reset_index(drop=True)
df_no_dupes_400 = df_no_dupes_400[df_no_dupes_400["LANG"].isin(["es", "en"])].reset_index(drop=True)

In [44]:
df_no_dupes_400["LANG"].value_counts()

LANG
en    97604
es       70
Name: count, dtype: int64

In [45]:
#Join all dataframes
df_final = pd.concat([df_no_dupes_100, df_no_dupes_200, df_no_dupes_201, df_no_dupes_300, df_no_dupes_400], ignore_index=True)
df_final

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,PROD_TYPE,MEDICAL_ATTN,VEHICLES_TOWED_YN,AGE,COMPDESC_len,LANG
0,FORD,EXPEDITION,2002.0,N,2013,N,0,0,SUSPENSION,111086.000000,1.0,routine service 2002 expedition dealership tol...,V,N,N,11.0,10,en
1,TOYOTA,CAMRY,2002.0,N,2005,N,0,0,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY:FI...",38500.000000,1.0,january 31 2005 dear toyota customer represent...,V,N,N,3.0,63,en
2,KIA,SORENTO,2019.0,N,2023,N,0,0,FUEL/PROPULSION SYSTEM,71775.184051,NaN,kia america sent letter june 30 2023 alerted o...,V,N,N,4.0,22,en
3,KIA,SORENTO,2004.0,N,2008,N,0,0,ENGINE AND ENGINE COOLING:ENGINE,61500.000000,1.0,response person filed complaint mine 2004 exac...,V,N,N,4.0,32,en
4,FORD,F-150,2005.0,Y,2005,N,0,0,"SERVICE BRAKES, HYDRAULIC",2000.000000,1.0,may 2 2006 accident happened december 19 2005 ...,V,N,N,0.0,25,en
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390369,HONDA,ACCORD,2005.0,N,2011,N,0,0,AIR BAGS,135000.000000,NaN,tl contact owns 2005 honda accord exl driving ...,V,N,N,6.0,8,en
390370,HONDA,ACCORD,2005.0,N,2013,N,0,0,AIR BAGS,126000.000000,1.0,tl contact owns 2005 honda accord hybrid conta...,V,N,N,8.0,8,en
390371,HONDA,ACCORD,2005.0,N,2012,N,0,0,STRUCTURE:BODY:TRUNK LID,182000.000000,1.0,tl contact owns 2005 honda accord hybrid conta...,V,N,N,7.0,25,en
390372,HONDA,ACCORD HYBRID,2005.0,N,2017,N,0,0,ENGINE,248000.000000,NaN,tl contact owns 2005 honda accord hybrid conta...,V,N,N,12.0,6,en


In [46]:
#save final dataframe
df_final.to_csv("df_final.csv", index=False)

In [47]:
df_final.describe()

,YEARTXT,FAILDATE,INJURED,DEATHS,MILES,AGE,COMPDESC_len
count,390374.000000,390374.000000,390374.000000,390374.000000,3.903740e+05,390374.000000,390374.000000
mean,2009.674366,2014.892162,0.048897,0.002516,7.165297e+04,5.243607,18.913688
std,6.198083,6.220939,0.381859,0.265423,9.205172e+04,3.845326,13.036948
min,2000.000000,2000.000000,0.000000,0.000000,-9.500000e+04,0.000000,3.000000
25%,2004.000000,2010.000000,0.000000,0.000000,4.950000e+04,2.000000,10.000000
50%,2009.000000,2016.000000,0.000000,0.000000,7.177518e+04,5.000000,16.000000
75%,2014.000000,2019.000000,0.000000,0.000000,7.582175e+04,8.000000,22.000000
max,2026.000000,2026.000000,99.000000,99.000000,9.999999e+06,26.000000,104.000000


In [61]:
df_final = df_final[(df_final["INJURED"] >= 0) & (df_final["INJURED"] <= 5)].reset_index(drop=True)
df_final = df_final[(df_final["DEATHS"] >= 0) & (df_final["DEATHS"] <= 5)].reset_index(drop=True)

In [62]:
df_final['compdesc_clean'] = (
    df_final['COMPDESC']
    .astype(str)
    .str.upper()
    .str.strip()
)

In [63]:
bad_values = ['UNKNOWN', 'NONE', 'NAN', 'OTHER/UNKNOWN', 'Other/I am not sure']

df_final = df_final[~df_final['compdesc_clean'].isin(bad_values)]

In [64]:
df_final['compdesc_lvl1'] = df_final['compdesc_clean'].str.split(':').str[0]

In [71]:
# map 1 yes in FIRE column
df_final['FIRE'] = df_final['FIRE'].map({'Y': 1, 'N': 0})
df_final['CRASH'] = df_final['CRASH'].map({'Y': 1, 'N': 0})


In [67]:
#drop columns prod_type, medical_attn, vehicles_towed_yn
df_final.drop(["PROD_TYPE", "MEDICAL_ATTN", "VEHICLES_TOWED_YN"], axis=1, inplace=True)

In [69]:
df_no_dupes_100['FIRE'].value_counts()

FIRE
N    95631
Y     1502
Name: count, dtype: int64

In [73]:
df_final['FIRE'].value_counts()

Series([], Name: count, dtype: int64)

In [72]:
df_final

,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,DEATHS,COMPDESC,MILES,OCCURENCES,CDESCR,AGE,COMPDESC_len,LANG,compdesc_clean,compdesc_lvl1
0,FORD,EXPEDITION,2002.0,NaN,2013,NaN,0,0,SUSPENSION,111086.000000,1.0,routine service 2002 expedition dealership tol...,11.0,10,en,SUSPENSION,SUSPENSION
1,TOYOTA,CAMRY,2002.0,NaN,2005,NaN,0,0,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY:FI...",38500.000000,1.0,january 31 2005 dear toyota customer represent...,3.0,63,en,"FUEL SYSTEM, GASOLINE:STORAGE:TANK ASSEMBLY:FI...","FUEL SYSTEM, GASOLINE"
2,KIA,SORENTO,2019.0,NaN,2023,NaN,0,0,FUEL/PROPULSION SYSTEM,71775.184051,NaN,kia america sent letter june 30 2023 alerted o...,4.0,22,en,FUEL/PROPULSION SYSTEM,FUEL/PROPULSION SYSTEM
3,KIA,SORENTO,2004.0,NaN,2008,NaN,0,0,ENGINE AND ENGINE COOLING:ENGINE,61500.000000,1.0,response person filed complaint mine 2004 exac...,4.0,32,en,ENGINE AND ENGINE COOLING:ENGINE,ENGINE AND ENGINE COOLING
4,FORD,F-150,2005.0,NaN,2005,NaN,0,0,"SERVICE BRAKES, HYDRAULIC",2000.000000,1.0,may 2 2006 accident happened december 19 2005 ...,0.0,25,en,"SERVICE BRAKES, HYDRAULIC","SERVICE BRAKES, HYDRAULIC"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390293,HONDA,ACCORD,2005.0,NaN,2011,NaN,0,0,AIR BAGS,135000.000000,NaN,tl contact owns 2005 honda accord exl driving ...,6.0,8,en,AIR BAGS,AIR BAGS
390294,HONDA,ACCORD,2005.0,NaN,2013,NaN,0,0,AIR BAGS,126000.000000,1.0,tl contact owns 2005 honda accord hybrid conta...,8.0,8,en,AIR BAGS,AIR BAGS
390295,HONDA,ACCORD,2005.0,NaN,2012,NaN,0,0,STRUCTURE:BODY:TRUNK LID,182000.000000,1.0,tl contact owns 2005 honda accord hybrid conta...,7.0,25,en,STRUCTURE:BODY:TRUNK LID,STRUCTURE
390296,HONDA,ACCORD HYBRID,2005.0,NaN,2017,NaN,0,0,ENGINE,248000.000000,NaN,tl contact owns 2005 honda accord hybrid conta...,12.0,6,en,ENGINE,ENGINE


In [75]:
df_final['compdesc_lvl1'].value_counts()

compdesc_lvl1
ELECTRICAL SYSTEM                                44083
POWER TRAIN                                      39340
AIR BAGS                                         35529
UNKNOWN OR OTHER                                 29715
ENGINE                                           28953
STRUCTURE                                        23272
STEERING                                         22909
FUEL/PROPULSION SYSTEM                           17621
VEHICLE SPEED CONTROL                            16883
SERVICE BRAKES                                   16854
ENGINE AND ENGINE COOLING                        14594
EXTERIOR LIGHTING                                12074
SERVICE BRAKES, HYDRAULIC                        12071
SUSPENSION                                       11781
ELECTRONIC STABILITY CONTROL (ESC)                8460
VISIBILITY                                        8026
FUEL SYSTEM, GASOLINE                             6849
VISIBILITY/WIPER                                  5